In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_openai import ChatOpenAI
import dotenv
import os
from langchain_chroma import Chroma

In [2]:
dotenv.load_dotenv()

True

In [3]:
embeddings = OllamaEmbeddings(
    model="jeffh/intfloat-multilingual-e5-small:f32",
)

In [4]:
llm = ChatOpenAI(
            api_key=os.getenv("OPENROUTER_API_KEY"),
            base_url="https://openrouter.ai/api/v1",
            model="openai/gpt-oss-20b:free"
        )

In [5]:
vectorstore = Chroma(
            collection_name="docs",
            embedding_function=embeddings,
            persist_directory="./chroma_docs"
        )

In [6]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

In [7]:
RETRIEVE_K = 20
CONTEXT_K = 5

In [8]:
from flashrank import Ranker 

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVE_K})
compressor = FlashrankRerank(top_n=CONTEXT_K, model='ms-marco-MiniLM-L-12-v2')
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

In [14]:
systemprompt = '''Ты профессиональный банковский помощник. Твоя задача давать клиенту правидивую и релевантную информацию по поводу банковских услуг.
                Тебе будут даны вопрос пользователя и контекст по 5 документам, в которых дана информация по теме вопроса
                Старайся брать ответы только из этих документов. Отвечай только на русском и соблюдай профессиональную этику'''
context = '''контекст: 1. {}
                       2. {}
                       3. {}
                       4. {}
                       5. {}'''
query = 'вопрос пользователя: {}'

prompt = ChatPromptTemplate(
    [MessagesPlaceholder("systemprompt"),
     MessagesPlaceholder("context"),
     MessagesPlaceholder("query")]
)

In [25]:
from typing import TypedDict, Optional
class RAGState(TypedDict):
    query: Optional[str]
    docs: Optional[list]
    answer: Optional[str]
    pages: Optional[list]

def input_node(state: RAGState):
    return state
    
def get_relevant_docs(state: RAGState):
    query = state['query']
    docs = compression_retriever._get_relevant_documents(f'query: {query}')
    pages = [doc.metadata['web_page'] for doc in docs]
    return {'docs': docs, 'pages': pages}

def get_answer(state: RAGState):
    msg = llm.invoke(prompt.invoke(
        {"systemprompt": SystemMessage(systemprompt),
        "context": SystemMessage(context.format([doc.page_content for doc in state['docs']])),
        "query": HumanMessage(query.format(state['query']))}
    ))
    answer = msg["messages"][-1].content
    return {'answer': answer}

In [26]:
from langgraph.graph import StateGraph

In [27]:
builder = StateGraph(RAGState)
builder.add_node("input", input_node)
builder.add_node("retriever", get_relevant_docs)
builder.add_node("output", get_answer)

builder.set_entry_point("input")
builder.add_edge("input", "retriever")
builder.add_edge("retriever", "output")
builder.set_finish_point("output")
graph = builder.compile()